In [1]:
import cv2
import matplotlib.pyplot as plt


img_original = cv2.imread('foto1.jpg')

img_avatar = cv2.imread('foto_personagens.png')

In [2]:
# a

def aplicar_filtros(img, nome_img):

    kernels = [3, 5, 11, 29]

    for k in kernels:

        # Filtro de média
        media = cv2.blur(img, (k, k))
        cv2.imwrite(f'{nome_img}_media_{k}x{k}.jpg', media)

        # Filtro Gaussiano
        gauss = cv2.GaussianBlur(img, (k, k), 0)
        cv2.imwrite(f'{nome_img}_gauss_{k}x{k}.jpg', gauss)

        # Filtro Mediana
        mediana = cv2.medianBlur(img, k)
        cv2.imwrite(f'{nome_img}_mediana_{k}x{k}.jpg', mediana)

        # Filtro Bilateral
        bilateral = cv2.bilateralFilter(img, k, 75, 75)
        cv2.imwrite(f'{nome_img}_bilateral_{k}x{k}.jpg', bilateral)

In [3]:
aplicar_filtros(img_original, "foto1")
aplicar_filtros(img_avatar, "foto2") # encontra-se na pasta imagens_resultantes

In [4]:
# b
import numpy as np

def adicionar_ruido_gaussiano(img, nome_img):

    # parâmetros do ruído
    media = 0
    sigma = 50   # valor alto → ruído severo

    # gerar matriz de ruído com mesma dimensão da imagem
    ruido = np.random.normal(media, sigma, img.shape)

    # adicionar ruído à imagem
    imagem_ruido = img + ruido

    # limitar valores entre 0 e 255
    imagem_ruido = np.clip(imagem_ruido, 0, 255)

    # converter para uint8
    imagem_ruido = imagem_ruido.astype(np.uint8)

    # salvar imagem
    cv2.imwrite(f'{nome_img}_ruido_gaussiano.jpg', imagem_ruido)

    print("Imagem com ruído salva.")

In [5]:
adicionar_ruido_gaussiano(img_original, "foto1")
adicionar_ruido_gaussiano(img_avatar, "foto2")

Imagem com ruído salva.
Imagem com ruído salva.


In [8]:
!pip install scikit-image

Defaulting to user installation because normal site-packages is not writeable


In [17]:
# c
import os
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# imagem original
img_original_gray = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

pasta = "imagens_resultantes/"

for arquivo in os.listdir(pasta):

    caminho = os.path.join(pasta, arquivo)

    img = cv2.imread(caminho)

    if img is None:
        continue

    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if img_gray.shape != img_original_gray.shape:
        img_gray = cv2.resize(img_gray, (img_original_gray.shape[1], img_original_gray.shape[0]))

    psnr = peak_signal_noise_ratio(img_original_gray, img_gray, data_range=255)
    ssim = structural_similarity(img_original_gray, img_gray, data_range=255)

    print(f"Imagem: {arquivo}")
    print(f"PSNR: {psnr:.2f}")
    print(f"SSIM: {ssim:.4f}")
    print()

/var/data/python/lib/python3.13/site-packages/skimage/metrics/simple_metrics.py:171: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


Imagem: foto1.jpg
PSNR: inf
SSIM: 1.0000

Imagem: foto1_bilateral_3x3.jpg
PSNR: 43.60
SSIM: 0.9888

Imagem: foto1_bilateral_5x5.jpg
PSNR: 40.47
SSIM: 0.9778

Imagem: foto1_media_11x11.jpg
PSNR: 26.41
SSIM: 0.7912

Imagem: foto1_media_29x29.jpg
PSNR: 21.51
SSIM: 0.6412

Imagem: foto1_mediana_3x3.jpg
PSNR: 41.23
SSIM: 0.9860

Imagem: foto2_gauss_3x3.jpg
PSNR: 11.16
SSIM: 0.3854

Imagem: foto2_gauss_5x5.jpg
PSNR: 11.18
SSIM: 0.3876

Imagem: foto2_gauss_11x11.jpg
PSNR: 11.28
SSIM: 0.3966

Imagem: foto2_mediana_29x29.jpg
PSNR: 11.36
SSIM: 0.4579

Imagem: foto2_ruido_gaussiano.jpg
PSNR: 11.32
SSIM: 0.1050

Imagem: foto_personagens.png
PSNR: 11.13
SSIM: 0.3834

Imagem: foto1_bilateral_11x11.jpg
PSNR: 35.69
SSIM: 0.9340

Imagem: foto1_bilateral_29x29.jpg
PSNR: 31.70
SSIM: 0.8957

Imagem: foto1_ruido_gaussiano.jpg
PSNR: 18.32
SSIM: 0.2052

Imagem: foto2_bilateral_3x3.jpg
PSNR: 11.14
SSIM: 0.3848

Imagem: foto2_mediana_5x5.jpg
PSNR: 11.16
SSIM: 0.3888

Imagem: foto2_mediana_11x11.jpg
PSNR: 11.24

In [ ]:
import time
from IPython.display import display, clear_output
from PIL import Image

fps = 30.0
width = 640
height = 480

cap = cv2.VideoCapture(0)

# definir resolução
cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('saida.avi', fourcc, fps, (width, height))

frame_count = 0
last_frame = None

try:
    print("Capturando frames... Pressione Ctrl+C para parar.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Erro na captura")
            break

        frame = cv2.flip(frame, 0)
        last_frame = frame

        # filtros escolhidos (exemplo)
        filtro_media = cv2.blur(frame, (29,29))
        filtro_bilateral = cv2.bilateralFilter(frame,5,75,75)

        # juntar imagens lado a lado
        resultado = cv2.hconcat([filtro_media, filtro_bilateral])

        # salvar vídeo original
        out.write(frame)

        # exibir frame a cada 30 frames (~1 segundo)
        if frame_count % 30 == 0:
            frame_rgb = cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB)
            display(Image.fromarray(frame_rgb))
            clear_output(wait=True)

        # salvar imagem a cada 100 frames
        if frame_count % 100 == 0:
            cv2.imwrite("captura.jpg", resultado)
            print(f"Frame {frame_count}: Imagem salva")

        frame_count += 1
        time.sleep(0.03)

except KeyboardInterrupt:
    print("\nCaptura interrompida pelo usuário")

finally:
    if last_frame is not None:
        cv2.imwrite("captura_final.jpg", last_frame)
    
    cap.release()
    out.release()
    print(f"Captura finalizada. Total de frames: {frame_count}")
    print("Vídeo salvo como: saida.avi")

error: OpenCV(4.13.0) /io/opencv/modules/highgui/src/window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'


In [36]:
cap = cv2.VideoCapture(0)

last_frame = None  # guarda o último frame válido

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        last_frame = frame

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        display(Image.fromarray(frame_rgb))
        clear_output(wait=True)

except KeyboardInterrupt:
    print("Interrompido! Salvando foto...")

finally:
    if last_frame is not None:
        cv2.imwrite("foto1.jpg", last_frame)
    
    cap.release()

NameError: name 'Image' is not defined